# MARIO disaggregation workflow

One API call for each workflow stage. Use a Python environment with this repository, MARIO and CVXLab importable. Copy `config/notebook.example.json` to `settings.json` beside this notebook and set your local paths.

MARIO parsing, aggregation and checkpoint analysis use `codex/mrio-memory`. The separated **build, solve and assembly** calls currently require MARIO's experimental `codex/explicit-workflow` checkout. Database disaggregation APIs are consolidated on `main`.

The demonstration reads Parquet, prepared inventories and an existing result. Preparation, solving and export cells are commented. They can be enabled for a fresh run with an appropriate input workbook and installed solver. All displayed sectors include intermediate uses and the available final-demand categories.

In [ ]:
import mario
import database_disaggregation as disagg

In [ ]:
from pathlib import Path
import json

In [ ]:
from IPython.display import FileLink, display

## Settings

The JSON contains your dataset paths and solver options. Reading it starts no model and changes no input files.

In [ ]:
settings = json.loads(Path("settings.json").read_text())
data_root = Path(settings["data_root"]).expanduser()

In [ ]:
baseline_path = data_root / settings["database_path"]
inventory_file = data_root / settings["workbook"]

In [ ]:
saved_run = data_root / settings["result_database"]
model_options = settings["split_model_options"]

In [ ]:
result = saved_run
benchmark = saved_run / "benchmarking.json"

## 1. Parse the MRIO database

Parquet is the practical starting point for this demonstration. For the original CSV inputs, replace the call with `mario.parse_gtap(path=baseline_path, input_format="csv", year=2023)` and update `database_path`. The following steps consume the same MARIO object.

In [ ]:
native_db = mario.parse_from_parquet(baseline_path, table="IOT", mode="flows", year=2023)

## 2. Aggregate the regions

Apply the shared native-to-model mapping. The original object remains available for preparing collected data at the native resolution.

In [ ]:
geography = disagg.Geography.read(data_root / settings["geography"])

In [ ]:
db = native_db.aggregate(None, levels="Region", region_aggregation=geography.mapping,
                         zero_output_epsilon=None, inplace=False)

## 3. Prepare the collected data

The preparation API applies the repository's collection, classification and reconciliation rules. It returns the matching regional database, workbook and audit. This demonstration uses the reviewed workbook already selected in the settings. Enable the next cell to prepare a new copy; the destination directory must be new.

In [ ]:
# prepared = disagg.prepare_inputs(database=native_db, data_root=data_root, geography=geography, output_dir="build/prepared_inputs")
# db, inventory_file = prepared.database, prepared.workbook

## 4. Read the MARIO inventories

The workbook contains the sector/parent identities, cost structures, production and trade constraints. It can be opened in Excel during the workshop.

In [ ]:
db.read_add_sectors_excel(inventory_file, read_inventories=True, split=True)

## 5. Build the optimization model

This stage constructs a CVXLab/CVXPY problem and writes its inputs; it runs no solver. The experimental staged API builds one problem for the supplied workbook. For grouped execution with checkpoints and failure continuation, use MARIO's `add_sectors(intermediate_steps=True, ...)` controller.

In [ ]:
# model = disagg.workflow.build_model(db, inventory_file, directory="build/notebook_run/model",
#                                     split_model_options=model_options)

## 6. Solve the optimization model

Run the selected solver and the existing numerical checks. Choose a manageable input case before enabling the model cells; this notebook does not rerun the full saved workshop case by default.

In [ ]:
# solution = disagg.workflow.solve_model(model, solver=settings["solver"],
#                                       solver_parameters=settings["solver_parameters"])

## 7. Assemble the disaggregated database

Reconstruct MARIO matrices from the validated solution. For a fresh result, clear the saved benchmark so it cannot be attached to another run.

In [ ]:
# result = disagg.workflow.assemble_database(solution)
# benchmark = None

## 8. Export the result

Save flows, independent production, inputs and run metadata to a new directory. This exported result is distinct from a resumable sequential checkpoint.

In [ ]:
# exported = disagg.workflow.export_database(result, "build/notebook_run/database", inputs=inventory_file)

## 9. Analyse the result

With the default cells, the dashboard reads and verifies the saved checkpoint and its matching benchmark. After a fresh assembly, it uses the in-memory result and checks its present sectors against the selected workbook. External comparisons are shown when supplied by a matching report; they are not fabricated for fresh runs.

In [ ]:
dashboard = disagg.workflow.build_dashboard(result, "results/MARIO_workshop_dashboard.html", inputs=inventory_file, benchmark=benchmark)
display(FileLink(dashboard))